In [1]:
from tqdm import tqdm 

In [2]:
import os
import json

def extract_audio_links(folder_path, output_file):
    """
    Extract audio links from JSON files in the specified folder
    and create a combined JSON with {filename: audio_link} format.
    """
    # Dictionary to store filename: audio link pairs
    audio_links = {}
    
    # Check if the folder exists
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' not found.")
        return None
    
    # Iterate through all files in the folder
    for filename in os.listdir(folder_path):
        # Only process JSON files
        if filename.endswith('.json'):
            file_path = os.path.join(folder_path, filename)
            
            try:
                # Read and parse the JSON file
                with open(file_path, 'r', encoding='utf-8') as file:
                    try:
                        data = json.load(file)
                        
                        # Extract the audio link
                        if (
                            'data' in data and 
                            'body' in data['data'] and 
                            'Audio' in data['data']['body']
                        ):
                            audio_link = data['data']['body']['Audio']
                            
                            # Only add to the dictionary if there's a valid audio link
                            if audio_link and audio_link != "No audio source found":
                                # Remove the .json extension from the filename
                                filename_without_extension = os.path.splitext(filename)[0]
                                audio_links[filename_without_extension] = audio_link
                                
                    except json.JSONDecodeError:
                        print(f"Error: Unable to parse JSON in file '{filename}'")
            except Exception as e:
                print(f"Error processing file '{filename}': {str(e)}")
    
    # Write the combined data to a new JSON file
    
    # output_file = f"{categ}_combined_audio_links.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(audio_links, f, ensure_ascii=False, indent=4)
    
    print(f"Successfully created '{output_file}' with {len(audio_links)} audio links.")
    return output_file


In [3]:
categ = "འཛམ་གླིང་།"
folder_path = f"./new_data/{categ}/"
file_name = f"{categ}_combined_audio_links.json"
output_file = folder_path + file_name

extract_audio_links(folder_path, output_file)

Successfully created './new_data/འཛམ་གླིང་།/འཛམ་གླིང་།_combined_audio_links.json' with 6673 audio links.


'./new_data/འཛམ་གླིང་།/འཛམ་གླིང་།_combined_audio_links.json'

### make DRI

In [4]:
mkdir ./new_data/audio/འཛམ་གླིང་།

### Extract Audio data

In [5]:
import requests
from bs4 import BeautifulSoup

def download_audio_from_rfa(url, output_filename):
    try:
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        # Send a GET request to the page
        response = requests.get(url, headers=headers)
        
        if response.status_code != 200:
            print(f"Failed to access the page. Status code: {response.status_code}")
            return
        
        # For audio streams, we might not need to parse the HTML
        # We can directly save the content as it's likely the audio file itself
        with open(output_filename, 'wb') as file:
            file.write(response.content)
        # print(f"Audio file downloaded successfully: {output_filename}")
        return True
    except Exception as e:
        print(f"Error processing: {str(e)}")
        return False

# # Usage
# audio_url = "https://voa-audio-ns.akamaized.net/vti/2025/02/08/6b264532-aacb-4a02-b533-08dd481ae9e5.mp3"
# output_filename = "downloaded_audio.mp3"

# download_audio_from_rfa(audio_url, output_filename)

In [6]:
def read_json(path, file_name):
    """
    
    """
    with open(path+file_name, 'r') as openfile:
        # Reading from json file
        Loaded_file = json.load(openfile)
        print(f"Successfully loaded: {file_name}")

    return Loaded_file


#### Laod audio json file

In [7]:
audio_file = read_json(folder_path, file_name)
print(len(audio_file))

Successfully loaded: འཛམ་གླིང་།_combined_audio_links.json
6673


#### Run each audio file and save in audio DIR

In [8]:
import requests
import urllib3
from tqdm import tqdm
import os

def download_audio_from_rfa(url, output_filename):
    try:
        # Disable SSL warnings if needed
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        
        # More comprehensive headers
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'audio/mpeg',
            'Connection': 'keep-alive'
        }
        
        # Increase timeout and allow redirects
        response = requests.get(
            url, 
            headers=headers, 
            timeout=30, 
            allow_redirects=True,
            verify=False  # Disable SSL verification if certificate issues persist
        )
        
        # Check if the request was successful
        response.raise_for_status()
        
        # Ensure the directory exists
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        
        # Save the file
        with open(output_filename, 'wb') as file:
            file.write(response.content)
        
        return True
    
    except requests.exceptions.RequestException as e:
        print(f"Error processing: {output_filename} : {str(e)}")
        return False

def batch_download_audio(audio_file, categ):
    audio_path = f"./new_data/audio/{categ}/"
    error_count = 0
    
    for name, audio_url in tqdm(audio_file.items()):
        output_filename = os.path.join(audio_path, f"{name}.mp3")
        
        success = download_audio_from_rfa(audio_url[0], output_filename)
        if not success:
            error_count += 1
        # print(audio_url)
        # break
    
    print(f"Total error count: {error_count}")

# Example usage
batch_download_audio(audio_file, categ)

 13%|█▎        | 885/6673 [19:25<1:02:01,  1.56it/s] 

Error processing: ./new_data/audio/འཛམ་གླིང་།/VOT_Tib_འཛམ་གླིང་།_1098.mp3 : ('Connection broken: IncompleteRead(2097152 bytes read, 1103943 more expected)', IncompleteRead(2097152 bytes read, 1103943 more expected))


 13%|█▎        | 886/6673 [19:26<1:26:29,  1.12it/s]

Error processing: ./new_data/audio/འཛམ་གླིང་།/VOT_Tib_འཛམ་གླིང་།_1099.mp3 : ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


 13%|█▎        | 893/6673 [19:40<2:23:59,  1.49s/it]

...

 62%|██████▏   | 4136/6673 [1:28:14<2:19:27,  3.30s/it]

Error processing: ./new_data/audio/འཛམ་གླིང་།/VOT_Tib_འཛམ་གླིང་།_4473.mp3 : ('Connection broken: IncompleteRead(2097152 bytes read, 520529 more expected)', IncompleteRead(2097152 bytes read, 520529 more expected))


 64%|██████▎   | 4250/6673 [1:32:05<1:05:36,  1.62s/it]

Error processing: ./new_data/audio/འཛམ་གླིང་།/VOT_Tib_འཛམ་གླིང་།_4589.mp3 : ('Connection broken: IncompleteRead(2097152 bytes read, 162338 more expected)', IncompleteRead(2097152 bytes read, 162338 more expected))


IOPub message rate exceeded.1:32:29<1:52:16,  2.79s/it]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [9]:
# Dictionary to store filename: audio link pairs
audio_links = {}

# Check if the folder exists
if not os.path.exists(folder_path):
    print(f"Error: Folder '{folder_path}' not found.")
count = 0
# Iterate through all files in the folder
for filename in os.listdir(folder_path):

    # Only process JSON files
    if filename.endswith('.json'):
        # file_path = os.path.join(folder_path, filename
        count += 1

print(f"Total File: {count}")

Total File: 8382


In [10]:
# Dictionary to store filename: audio link pairs
audio_links = {}
audio_path = f"./new_data/audio/{categ}/"


# Check if the folder exists
if not os.path.exists(audio_path):
    print(f"Error: Folder '{audio_path}' not found.")
audio_count = 0
# Iterate through all files in the folder
for filename in os.listdir(audio_path):

    # Only process JSON files
    if filename.endswith('.mp3'):
        # file_path = os.path.join(folder_path, filename
        audio_count += 1

print(f"Total audio File: {audio_count}")

Total audio File: 6049


In [11]:
file_with_audio = 6673 

print(f"This is for {categ}")
print(f"Total file we had was {count}")
print(f"file having audio link is {file_with_audio}")
print(f"Audio successfully extracted {audio_count} ")
print(f"Total audio file lost in error {file_with_audio - audio_count} as {round((file_with_audio - audio_count)/file_with_audio * 100)}%")
# print(f"Total file we had was  and now we have successfully extracted {5461 }")

This is for འཛམ་གླིང་།
Total file we had was 8382
file having audio link is 6673
Audio successfully extracted 6049 
Total audio file lost in error 624 as 9%
